In [ ]:
from sklearn import datasets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
iris = datasets.load_iris()
X = iris.data  # get the samples
t = iris.target  # get the labels

# Convert to a DataFrame for a nicer view
df_iris = pd.DataFrame(data=np.c_[X, t], columns=iris.feature_names + ['target'])
display(df_iris.head())

In [ ]:
from sklearn import datasets

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, interactive_output

# Introduction

In this course we will work with algorithms for analyzing datasets, such as data coming from industrial and commercial processes, images, and text or structured documents. Exploratory analysis is closely related to data mining, knowledge discovery, and processes such as **KDD** (Knowledge Discovery in Databases) and **CRISP-DM**. Typical data-analysis projects can be divided into several phases: preparation, preprocessing, analysis, and post-processing. In this section we will focus mainly on the analysis and preprocessing phases.

## Data analysis

Analysis consists of applying computational algorithms to process and analyze large datasets. The algorithms and methods used are drawn from other scientific disciplines such as statistics, machine learning (as in this course), pattern recognition, systems theory, operations research, and artificial intelligence.

### Data-analysis project

Data-analysis projects can be divided into several phases.

1. **Preparation** — Data collection and selection, feature generation.
2. **Preprocessing** — Cleaning, filtering, imputation, correction, standardization.
3. **Analysis** — Visualization, correlation, classification, regression, forecasting, clustering.
4. **Post-processing** — Interpretation, documentation, evaluation.

In this **Machine Learning** course we will present some basic ideas from the preprocessing and analysis phases that make it possible to perform a "quick" exploration and set the stage for a deeper analysis. Our study will focus on selected preprocessing and analysis methods for both structured and unstructured data.

## Datasets

A dataset is a collection of elements used to answer questions or make decisions. A dataset usually consists of a set of observations, where each observation is made up of a group of variables; the observations are the individual data points and the variables are the characteristics (features) of the data.

Features can be defined in different spaces and contexts. Some of the main types are:
- **Numeric**: this type of data represents numbers. It can be discrete (integers) or continuous (decimal values). Examples include age, height, weight, and price.
- **Categorical**: this type of data represents categories. It can be nominal (categories with no inherent order) or ordinal (categories with an inherent order). Examples of categorical attributes include gender, eye color, and state of residence.
- **Text**: this type of data represents written documents. It can be unstructured (free-form text) or structured (text following a specific format). Examples include product reviews, customer-support tickets, and social-media posts.
- **Date and time**: this type of data represents dates and times. It can be absolute (a specific date and time) or relative (a date or time relative to another). Examples include date of birth and the date and time an order arrived.
- **Spatial data**: this type of data represents locations. It can consist of points (a single location), routes (a series of connected locations), or polygons (a closed area). Examples include GPS coordinates, street addresses, and borders.

### The Iris dataset

The Iris dataset is one of the most widely used datasets for introducing the basic concepts of data analysis and machine learning. It was originally created in 1935 by the American botanist Edgar Anderson, who studied the geographic distribution of Iris flowers on the Gaspé Peninsula in Quebec, Canada.
The Iris dataset contains 150 measurements of Iris flower samples: 50 from each of the three species Iris Setosa, Iris Virginica, and Iris Versicolor. For each of the 150 flowers, four numeric features chosen by Anderson were measured: the length and width of the sepals and petals, in centimeters. We can access the Iris dataset from the `sklearn` library through the `datasets` module.

In [ ]:
iris = datasets.load_iris()
X = iris.data  # get the samples
Y = iris.target  # labels
X.shape  # 150 samples with 4 features each

In [ ]:
iris.feature_names  # description of each feature

In [ ]:
iris.target_names  # class names

In [ ]:
iris.target

In [ ]:
t = np.array([iris.target_names[i] for i in iris.target])
t = t.reshape((-1, 1))

In [ ]:
df = pd.DataFrame(columns=iris.feature_names + ['class', 'description'],
                  data=np.hstack((X.astype(np.float32),
                  iris.target.reshape((-1, 1)).astype(np.int32), t)))  # for a fancier view

In [ ]:
df.head()

In [ ]:
for c in df.columns[:-1]:
  if c != 'class':
    df[c] = df[c].astype(np.float32)
  else:
    df[c] = df[c].astype(np.int32)

In [ ]:
df.head()  # show the first 5 samples

In [ ]:
df.dtypes

In [ ]:
# plot sepal width against sepal length
fig, axis = plt.subplots(figsize=(6, 6))
for cls in set(df['class'].values):
  df_cls = df[df['class'] == cls]
  axis.scatter(df_cls['sepal length (cm)'],
               df_cls['sepal width (cm)'],
               label=iris.target_names[cls])
plt.legend()

In [ ]:
sum(df.isna()['sepal length (cm)'].values)

In [ ]:
df['class'].values

In the dataset each of the 150 Iris flowers is an object, each of the four dimensions is a **feature** or **attribute**, while the **class** is the family it belongs to (in the figure we use it to color each object).

Below we list some of the typical questions we try to answer through an initial analysis of the data:

- Which data points might contain errors or false class assignments?
- What error is introduced when rounding the data to one decimal place?
- What is the correlation between petal length and petal width?
- Which pair of dimensions is the most highly correlated?
- What is the maximum sepal width of a flower?
- What would be expected of a flower with a sepal width of 1.8 cm?
- To which species would an instance with a sepal width of 1.8 cm belong?
- Do the three species contain subgroups that can be identified from the data?

In [ ]:
df.describe()

## Establishing relationships between data

Mathematics lets us establish relationships between real numbers; the most commonly used relations are of the form $>, <, \geq, \leq, =, \neq$. In data analysis, establishing relationships between elements of a dataset is critically important. For that reason we devote a couple of sections to reviewing some concepts related to this topic.

Data types (nominal, ordinal, interval, ratio) must be taken into account, because certain mathematical operations are appropriate only for specific types. Numeric data can be represented using sets, vectors, or matrices. When working with numeric data, relationships are often established through dissimilarity/distance measures (Lebesgue, Minkowski, Bray-Curtis, Canberra, etc.) or similarity measures (such as cosine, Dice, Jaccard, Tanimoto). Sequences can be analyzed using sequence relations (such as Hamming distance or edit distance).

### Scales
Numeric measurements can have different semantic meanings, and the relations and statistics we can apply will differ accordingly. The following table shows some of the most common examples.

Scale   | Operations | Example | Statistics
-------- |-----------|-------------|------------
Ratio     | $*, /$      | 21 years, 273K      | Mean
Interval |$+,-$        | 2000 BC, 35ºF    | Mean
Ordinal   |$>,<$        | A, B+, B, C+, C  | Median
Nominal   |$=, \neq$    | Red, green, blue | Mode
----------------------------------------------------
**Scales for different data types**

From the table we can read, for example, that for nominal data (bottom row) only equality and inequality tests are valid. The values of a nominal feature can be represented by the mode, defined as the most frequent value. For interval attributes the addition and subtraction operations (second row) are valid. Interval-scaled features have arbitrary zero points, as in the Anno Domini dating system or temperatures in degrees Celsius (centigrade) or Fahrenheit, so it makes no sense, for example, to say that 40 °C is twice 20 °C. The data of an interval-scaled feature — for example, a set of values $X = \{x_1, \dots, x_n\}$ — can be represented by the mean (it would be useful to review your probability and statistics material).

In [ ]:
df.describe()  # we can get a basic summary of some statistics using the describe method

In [ ]:
np.bincount(df['class'])

## Matrix representation

In general, a dataset made up only of numeric features is represented as

$$X=\{x_1,x_2,\dots,x_n\}$$

where $n$ is the number of objects/elements in $X$, and each element is a $p$-dimensional feature vector, where $n$ and $p$ are positive integers. For $p = 1$ we call $X$ a scalar dataset. When $p>1$ the dataset is represented as a matrix.

$$\left(
  \begin{array}{cccc}
  x_1^{(1)} & x_1^{(2)}&\dots&x_1^{(p)}\\
  x_2^{(1)} & x_2^{(2)}&\dots&x_2^{(p)}\\
  \vdots & \vdots & \dots & \vdots \\
  x_n^{(1)} & x_n^{(2)}&\dots&x_n^{(p)}\\
  \end{array}
\right)$$

where each vector $x_1,\dots, x_n$ is a row vector with $p$ elements. Loosely, we can use datasets and matrices as equivalent representations. Each row of the data matrix corresponds to one element of the dataset. Each column of the data matrix corresponds to a feature across all elements of the dataset. The $i$-th feature or component is denoted by $x^{(i)}, i = 1,\dots,p$.

The associated class or output is represented as a vector $Y=\{y_1, y_2,\dots, y_n\}$, where each $y_i$ is the class associated with each object $x_i \in X$.

## Relations

Given a set of (abstract) elements, not necessarily referring to numeric feature vectors,

$$O=\{o_1,o_2,\dots,o_n\}$$

sometimes there is no feature-vector representation available for the objects $o_k, k = 1, \dots , n$, so conventional feature-based data-analysis methods are not applicable (at least not directly). Instead, the relation between all pairs of objects can often be quantified and written as a square matrix.

$$R=\left(
  \begin{array}{cccc}
  r_{11} & r_{12}&\dots&r_{1n}\\
  r_{21} & r_{22}&\dots&r_{2n}\\
  \vdots & \vdots & \dots & \vdots \\
  r_{n1} & r_{n2}&\dots&r_{nn}\\
  \end{array}
\right) \in \mathbb{R}^{ n \times n} $$

where each relation value $r_{ij}, i, j = 1, \dots, n$ in $R$ may refer to a degree of similarity, dissimilarity, compatibility, incompatibility, proximity, or distance between the pair of objects $o_i$ and $o_j$, with $r_{ij}=r_{ji}$ for all $i,j = 1, \dots,n$. The relation matrix $R$ can be defined manually or computed from the features. If the numeric features $X$ are available, then $R$ can be computed from $X$ using an appropriate function $f : \mathbb{R}^p  \times \mathbb{R}^p  \rightarrow R$. For example, a relation matrix for Iris could be defined manually by a botanist who visually compares flowers and assigns numeric relations between pairs of flowers, or $R$ could be computed from the sepal and petal lengths and widths. Below we present some similarity and distance relations.

### Dissimilarity / distance relations

A function $d$ is called a dissimilarity or distance measure if for all $x, y \in \mathbb{R}^p$,

$$\begin{array}{l}
d(x,y)=d(y,x)\\
d(x,y)=0 \Leftrightarrow x=y\\
d(x,z)\leq d(x,y)+d(y,z)
\end{array}$$

From the previous axioms it follows that $d(x,y) \geq 0$. A class of dissimilarity measures is defined by the norm $\lVert . \rVert$ of $x - y$, so that

$d(x,y)=\lVert x-y \rVert$

A function $\lVert . \rVert : \mathbb{R}^p  \times \mathbb{R}^p  \rightarrow R$ is a norm if and only if

$$\begin{array}{l}
\lVert x \rVert = 0 \Leftrightarrow x=(0,\dots,0) \\
\lVert a.x \rVert = |a| \cdot \lVert x \rVert  ~\forall_a\in\mathbb{R}, x \in \mathbb{R}^p\\
\lVert x+y \rVert = \lVert x \rVert+\lVert y \rVert ~\forall_{x,y} \in \mathbb{R}^p
\end{array}$$

For example, the so-called hyperbolic norm, given by the following equation,

$$
\lVert x \rVert_h = \Pi_{i=1}^p x^{(i)}
$$

is not a norm, since for instance the condition $\lVert x \rVert = 0 \Leftrightarrow x=(0,\dots,0)$ does not hold when $x=(0,1)$, because $\lVert x \rVert_h=0$ even though $x \neq (0,0)$.

The classes of norms used most frequently are matrix norms and Lebesgue or Minkowski norms. A matrix norm is defined as

$$\lVert x \rVert_A=\sqrt{x A x^T}$$

for a matrix $A \in \mathbb{R}^{n \times n}$. Some important cases for the matrix $A$ are defined in the following table.

|Name|A||
|----|---|--|
|Euclidean| $$\left(
  \begin{array}{cccc}
  1 & 0 &\dots&0\\
  0 & 1 &\dots& 0\\
  \vdots & \vdots & \ddots & \vdots \\
  0 &  0 &\dots & 1\\
  \end{array}
\right)$$ ||
|Frobenius or Hilbert-Schmidt| $$\left(
  \begin{array}{cccc}
  1 & 1 &\dots& 1\\
  1 & 1 &\dots& 1\\
  \vdots & \vdots & \ddots & \vdots \\
  1 &  1 &\dots & 1\\
  \end{array}
\right)$$ ||
|Diagonal | $$\left(
  \begin{array}{cccc}
  d_1 & 0 &\dots&0\\
  0 & d_2 &\dots& 0\\
  \vdots & \vdots & \ddots & \vdots \\
  0 &  0 &\dots & d_p\\
  \end{array}
\right)$$ | each attribute $i$ is weighted by $d_i$ |
|Mahalanobis | $\textit{cov}^{-1}X$ | Adapts the weighting of individual features according to the observed statistics. |

The Lebesgue or Minkowski norm is defined as

$$
\lVert x \rVert_{\alpha} = \sqrt[\alpha]{\sum_{j=1}^p |x^{(j)}|^{\alpha}}
$$

with $\alpha \in  \mathbb{R} \setminus \{0\}$, which equals the generalized mean except for a constant factor $\sqrt[\alpha]{n}$. Important special cases of the Lebesgue or Minkowski norm are summarized in the following table.

|Name|definition||
|------|---------|-------|
|Infimum $\alpha \rightarrow -\infty$| $$\lVert x \rVert_{-\infty}= \min_{j=1,2\dots p} x^{(j)} $$ | |
|Manhattan (city block) $\alpha=1$|$$
\lVert x \rVert_{\alpha} = \sum_{j=1}^p |x^{(j)}|
$$ | |
|Euclidean|$$
\lVert x \rVert_{2} = \sqrt[\alpha]{\sum_{j=1}^p (x^{(j)})^{2}}
$$||
|Supremum $\alpha \rightarrow \infty$| $$\lVert x \rVert_{\infty}= \max_{j=1,2\dots p} x^{(j)} $$ | |

Another frequently used dissimilarity is the Hamming distance, defined as:

 $$d_H(x,y)= \sum_{i=1}^p \rho(x^{(i)},y^{(i)}) $$

 where

 $\rho(x,y)=\begin{cases}
 0 \text{ if } x=y\\
 1 \text{ otherwise}
 \end{cases}$

 Note that the Hamming distance counts the number of feature values that do not match. For binary features, the Hamming distance equals the Manhattan distance $d_H (x, y) = x - y$. Observe, however, that the Hamming distance is not associated with a norm, because the condition $\lVert a.x \rVert = |a| \cdot \lVert x \rVert  \forall_a\in\mathbb{R}, x \in \mathbb{R}^p$ does not hold. Variants of the Hamming distance use modified functions $\rho$ to specify similarities between individual features. For example, if the features are (nominal-scale) text documents, then $\rho$ could be smaller for pairs with similar content and larger for pairs of documents with very different content.

## Similarity measures

A function $s$ is called a similarity or proximity measure if for all $x, y \in \mathbb{R}^p$

$$\begin{array}{l}
s(x,y)=s(y,x)\\
s(x,x)\leq s(x,y)\\
s(x,z)\geq 0
\end{array}$$

Additionally, if $s(x,y)=0$ it is called a normalized similarity function.
Any dissimilarity measure $d$ can be used to define a similarity measure $s$ and vice versa — for example, using a positive, monotonically decreasing function $f$ with $f(0) = 1$ such as

$$s(x,y)=\frac{1}{1+d(x,y)}$$

Let us first consider similarities between binary feature vectors. A pair of binary feature vectors can be considered similar if many of their 1s coincide. This conjunction can be represented by multiplication, so the dot product of the feature vectors is a reasonable candidate for a similarity measure. Also, for nonnegative real-valued features $x,y \in (\mathbb{R^+})^p$, similarity measures can be defined based on dot products normalized in different ways:

- Cosine similarity

$$s(x,y)=\frac{\sum_{i=1}^p x^{(i)}y^{(i)}}{\sqrt{\sum_{i=1}^p (x^{(i)})^2 \sum_{i=1}^p (y^{(i)})^2}}$$

- Overlap similarity

$$s(x,y)=\frac{\sum_{i=1}^p x^{(i)}y^{(i)}}{\min\left(\sum_{i=1}^p (x^{(i)})^2 ,\sum_{i=1}^p (y^{(i)})^2\right)}$$

- Dice similarity

$$s(x,y)=\frac{2\sum_{i=1}^p x^{(i)}y^{(i)}}{\sum_{i=1}^p (x^{(i)})^2 +\sum_{i=1}^p (y^{(i)})^2}$$

- Jaccard (Tanimoto)

$$s(x,y)=\frac{\sum_{i=1}^p x^{(i)}y^{(i)}}{\sum_{i=1}^p (x^{(i)})^2 +\sum_{i=1}^p (y^{(i)})^2- \sum_{i=1}^p x^{(i)}y^{(i)}}$$

These expressions are not defined for vectors in which all features are zero, because the denominators are zero; the similarity must therefore be defined explicitly for this case, for example as zero.

## Relations on sequences

In this section we consider measures that apply to sequences of feature values or feature vectors — for example, sequences of daily temperature values, text documents (sequences of alphanumeric characters), or sequences of visited web pages. Formally, such sequences could be viewed as feature vectors, but it is more appropriate to explicitly consider their sequential nature, the fact that each element of the sequence refers to the same feature, and the ability to compare sequences of different lengths.

We use a function $\rho$ to compare individual pairs of sequence elements. One example is the binary inequality function used in the Hamming distance (so it can be used as a relation for sequences of equal length).

To compute the relation between sequences of different lengths, neutral elements such as zeros or blank characters can be added to the shorter sequence. Depending on the alignment of the subsequences, lower Hamming distances can be achieved by prepending or optimally inserting neutral elements. This is the idea behind the Levenshtein distance, or edit distance, which determines the minimum number of editing operations (insert, delete, or change a sequence element) needed to transform one sequence into another. We denote by $L_{ij}(x, y)$ the edit distance between the first $i$ elements of $x$ and the first $j$ elements of $y$, and recursively define the edit distance as:

$$L_{ij}=\begin{cases}
  i \text{ if } j=0\\
  j \text{ if } i=0\\
  \min\{L_{i-1,j}+1,L_{i,j-1}+1, L_{i-1,j-1}+1+\rho(x^{(i)},y^{(i)})\}
\end{cases}$$

The first two cases consider empty sequences and terminate the recursion (either $x$ or $y$). In the third case, the three arguments of the minimum operator correspond to the three editing operations: insert, delete, and change. To compute $L_{ij}$ we need to compute every $L_{ij}, i = 1 ,\dots, p_x , j = 1,\dots, p_y$. The recursive implementation is inefficient, but a dynamic-programming solution can be implemented as follows:

In [ ]:
def levenshtein_distance(x, y):
    m = len(x)  # number of elements in sequence x
    n = len(y)  # number of elements in sequence y

    # Create an (m+1) x (n+1) matrix to store the subproblems
    # We use m+1 and n+1 to add an empty prefix at the beginning
    dp = np.zeros((m + 1, n + 1), dtype=int)

    # Initialize the first row and column of the matrix
    dp[:, 0] = np.arange(m + 1)
    dp[0, :] = np.arange(n + 1)

    # Compute the Levenshtein distance for the remaining subproblems
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if x[i - 1] == y[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    # Return the final result and the resulting matrix
    return dp[m][n], dp

In [ ]:
x, y = "INFOTEC", "INFORMACION"
d, L = levenshtein_distance(x, y)
print(f"The number of edits required to transform {x} into {y} is: {d}")

The following table shows the edit-distance matrix $L$ for the alphanumeric character sequences *INFOTEC* and *INFORMACION*. Each element of the matrix is computed as the minimum of its top neighbor plus one, its left neighbor plus one, and its top-left diagonal neighbor plus the corresponding character distance between the two sequences. The bottom-right value is the edit distance between the full sequences. This means we need at least that many editing operations to convert the sequence *INFOTEC* into the sequence *INFORMACION* and vice versa.

In [ ]:
pd.DataFrame(L, columns=[" "] + list(y), index=[" "] + list(x))

If we compute the minimum of each column, we can see that one edit must be performed each time there is an increase.

In [ ]:
np.min(L, axis=0)  # minimum of each column

Note that the first five characters are equal, so the value $d$ stays at 0 (we say five because the first character would be the empty string). For the sixth character we must make one edit to change *T* to *R* (or vice versa); for the characters *M* ($d=1$) and *A* ($d=3$) we either insert (to transform INFOTEC into INFORMACION) or delete (to go the other way). The value stays at 3 because both strings contain the character *C*; for the characters *I* ($d=4$), *O* ($d=5$), and *N* ($d=6$) an insertion or deletion is again required in each case.

## Sampling and quantization

In the previous section we considered finite discrete sequences of features or feature vectors. In many cases, such a sequence is obtained by sampling a continuous signal $x(t)$ with a fixed sampling period $T$ — for example, measuring wind speed every minute — so that we obtain a sequence called a time series

$$x(t)=x(k.T),~~~ k=1,\dots,n$$

Suppose the signal were defined by the function

$f(t)= e^{-t/3} \sin(2\pi t), \text{ for } 0 < t < 10$

In [ ]:
t = np.arange(0.1, 10, 0.01)  # the "original" signal
xt = np.exp(-t / 3) * np.sin(2 * np.pi * t)
plt.plot(t, xt)

The time series we can access will contain only individual samples and not the continuous signal, so it can cover only part of the information contained in the signal (not the infinite points contained in a continuous signal). With this in mind, we can see that as we increase the value of $T$, the quality of the time series becomes more and more imprecise until it no longer provides useful information.

In [ ]:
def sampling(T=0.7):
  fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
  t = np.arange(0.1, 10, 0.01)  # the "original" signal
  xt = np.exp(-t / 3) * np.sin(2 * np.pi * t)
  tm = np.arange(0.1, 10, T)  # sampling frequency
  xtm = np.exp(-tm / 3) * np.sin(2 * np.pi * tm)
  axes[0].plot(t, xt, label="Original signal")
  axes[1].plot(tm, xtm, label=f"Time series T={T}", marker='.')
  axes[0].legend()
  axes[1].legend()
  plt.title("Effect of increasing T")
  plt.show()

In [ ]:
interact(sampling, T=(0.1, 1.5, 0.1));

Any finite continuous signal can be represented as the sum of periodic signals with different frequencies (Fourier series). A signal $x(t)$ is called band-limited if the maximum frequency $f_{\max}$ of these periodic signals is finite, so that the Fourier spectrum is $|x(j2\pi f)| = 0$ for $f > f_{\max}$. If this signal is sampled with a sampling period smaller than $Ts = 1/(2 f_{\max})$, or equivalently a sampling frequency greater than $f_s = 2 f_{\max}$, then the original signal can be completely reconstructed from the (infinite) time series. This is known as Shannon's sampling theorem. The condition $T \leq Ts$ (or $f \geq f_s$) is called the Nyquist condition. In practical data-analysis projects, often only sampled data is available and not the original signals. Since only the sampled data is available, it is not possible to determine whether the sampling was performed according to the Nyquist condition. It is therefore often useful to discuss this topic with the data providers.

### Quantization

The first part of this section considered discretization in time, called sampling. Next we consider discretization of the data values, called quantization.

Quantization applies to analog values that are digitized with finite precision, as well as to digital values whose precision is reduced to save memory or speed up data transmission. Quantization maps a continuous interval $[x_{\min}, x_{\max}]$ to a set of discrete values $\{x_1,\dots, x_q\}$, where $q$ is the number of quantization levels. Each quantized value can be represented as a binary number of $b = \lceil\log_2 q\rceil$ bits, for example. Each continuous value $x \in  [x_{\min}, x_{\max}]$ can be translated to a quantized value $x_k \in \{x_1, \dots , x_q\}$ or to an index $k \in \{1,\dots,q\}$ by rounding.

$$\frac{\frac{x_{k-1}+x_k}{2}+x_1}{x_q-x_1} \leq \frac{x-x_{\min}}{x_{\max}-x_{\min}}< \frac{\frac{x_k+x_{k+1}}{2}+x_1}{x_q-x_1} $$

The quantization process causes a quantization error. The quantization error is related to the number of bits used to represent a numeric datum. A binary number with $b$ bits can represent integers between $x_1 = 0$ and $x_q = 2^b-1$. The relative quantization error can be estimated as $|e/(x_q - x_1)| \leq 100\%/2/(2^b-1) \approx b+1$. For typical values of $b \geq 8$, this quantization error can often be ignored in practice if the bounds $x_{\min}$ and $x_{\max}$ are chosen appropriately. If the range $x_{\max}-x_{\min}$ is much larger than the actual variation of the data, then the quantized data may appear constant or possibly show sudden jumps caused by the boundaries of the quantization levels.

In [ ]:
# install https://contrib.scikit-learn.org/category_encoders/
!pip install category_encoders

In [ ]:
# NOTE: illustrative example. It requires `import category_encoders as ce`
# and a categorical column (here 'size') present in the DataFrame.
import category_encoders as ce
oc = ce.ordinal.OrdinalEncoder(cols=['size'])
df_enc = oc.fit_transform(df)
df_enc